# Gavin Angeloff, PyTorch Implementation
**Converted from TensorFlow/Keras to PyTorch**

### **CREDIT: Google Gemini Converted the Tensorflow code to PyTorch**

### Dataset I trained on: https://www.kaggle.com/datasets/gpiosenka/cards-image-datasetclassification/data

### Import Statements & Device Configuration

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd
import os
# Class for training
from model import CardClassifier

# Check Pytorch Version
print(f"PyTorch Version: {torch.__version__}")

# Device Configuration
# PyTorch supports 'mps' for macOS GPU acceleration (Metal Performance Shaders)
# and 'cuda' for NVIDIA GPUs.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" # mac metal performance shaders: https://developer.apple.com/metal/pytorch/
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

PyTorch Version: 2.9.1+cpu
Using device: cpu


### Dataset Importing & Transforms
In PyTorch, data augmentation is handled via `transforms`.

In [2]:
# Constants
BATCH_SIZE = 32
IMG_SIZE = (200, 200)
NUM_CLASSES = 53

# Training Transforms (Data Augmentation)
# Matches Keras: RandomFlip, Translation/Rotation/Zoom (via RandomAffine), Contrast
train_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    # Simulate Keras RandomCrop if strictly needed, otherwise Resize is often sufficient for fixed input
    # transforms.RandomCrop(IMG_SIZE, padding=4), 
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(
        degrees=0.5 * 360,          # Matches RandomRotation(0.5)
        translate=(0.5, 0.5),       # Matches RandomTranslation(0.5, 0.5)
        scale=(0.5, 1.5)            # Matches RandomZoom(0.5) roughly
    ),
    transforms.ColorJitter(contrast=0.5), # Matches RandomContrast(0.5)
    transforms.ToTensor(),          # Converts to (C, H, W) and scales [0, 1]
])

# Validation Transforms (No Augmentation)
valid_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
])

# Load Datasets
print("ds_train:")
# Ensure the 'card_data' folder exists in your directory
train_dataset = datasets.ImageFolder(
    root='card_data/train',
    transform=train_transforms
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"Found {len(train_dataset)} files belonging to {len(train_dataset.classes)} classes.")

print("ds_valid:")
valid_dataset = datasets.ImageFolder(
    root='card_data/valid',
    transform=valid_transforms
)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Found {len(valid_dataset)} files belonging to {len(valid_dataset.classes)} classes.")

ds_train:
Found 7624 files belonging to 53 classes.
ds_valid:
Found 265 files belonging to 53 classes.


### Model Definition
Mimicking the Sequential Keras model structure.

In [3]:
# MOVED CLASS TO model.py

model = CardClassifier(num_classes=NUM_CLASSES).to(device)
print(model)

CardClassifier(
  (bn1): BatchNorm2d(3, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3a): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3b): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn_head): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (global_avg_pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc): Linear(in_features=256, out_features=53, bias=True)
)


### Saving Functions for Training

In [4]:
def save_model(model, epoch):
    # Create directory if it doesn't exist
    os.makedirs('trained_model1', exist_ok=True)

    # Save the model state dict (weights)
    torch.save(model.state_dict(), f'trained_model1/model1epoch{epoch}.pth')
    print(f"Model epoch {epoch} saved")

def save_train_plot(history, epoch):
    # Plot learning curves
    history_frame = pd.DataFrame(history)

    plt.figure(figsize=(12, 5))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(history_frame['loss'], label='loss')
    plt.plot(history_frame['val_loss'], label='val_loss')
    plt.title('Loss')
    plt.legend()

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history_frame['accuracy'], label='accuracy')
    plt.plot(history_frame['val_accuracy'], label='val_accuracy')
    plt.title('Accuracy')
    plt.legend()

    # Superplot title
    plt.suptitle(f"Model1 at {epoch} Epochs")

    # Create directory if it doesn't exist
    os.makedirs('model1_plots', exist_ok=True)

    plt.savefig(f"model1_plots/train_plot{epoch}.svg", format='svg', bbox_inches='tight')
    print(f"Plot epoch {epoch} saved")

### Training Setup

In [5]:
# Optimizer matches: tf.keras.optimizers.Adam(epsilon=0.001)
optimizer = optim.Adam(model.parameters(), lr=0.001, eps=0.001)

# Loss matches: sparse_categorical_crossentropy
# In PyTorch, CrossEntropyLoss expects logits (raw outputs) and class indices
criterion = nn.CrossEntropyLoss()

epochs = 3000
save_every = 50

### Training Loop

In [ ]:
history = {
    'loss': [],
    'accuracy': [],
    'val_loss': [],
    'val_accuracy': []
}

print("Starting training...")

for epoch in range(epochs):
    print(f"Epoch: {epoch}")
    model.train() # Set model to training mode
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    # Training Phase
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / total_train
    epoch_acc = correct_train / total_train
    
    # Validation Phase
    model.eval() # Set model to evaluation mode
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad(): # No gradients needed for validation
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = correct_val / total_val
    
    # Update History
    history['loss'].append(epoch_loss)
    history['accuracy'].append(epoch_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_accuracy'].append(epoch_val_acc)
    
    print(f"Epoch {epoch+1}/{epochs} - "
          f"loss: {epoch_loss:.4f} - acc: {epoch_acc:.4f} - "
          f"val_loss: {epoch_val_loss:.4f} - val_acc: {epoch_val_acc:.4f}")
    
    # Save every x epochs model
    if epoch % save_every == 0:
        save_model(model, epoch)
        save_train_plot(history, epoch)

    # Save last epoch model
    if epoch == epochs-1:
        save_model(model, epoch)
        save_train_plot(history, epoch)

#650 epochs was approx 4 hours

Starting training...
Epoch: 0
